# 02 — Phase 2: Conflict Experiment

Run every **conflict** prompt through the model and trace the logit-lens
curve (logit_diff at each layer) to find the **phase transition layer** —
the point where the model "makes up its mind".

**Outputs**
- `data/results/phase2/layer_curves_{prompt_id}.npy` — (n_layers,) per prompt
- `data/results/phase2/phase2_summary.csv` — phase_transition layer per prompt
- Figures saved to `figures/`

**Expected runtime:** ~10 minutes on M5 Pro

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from circuit_conflict.utils import load_model, get_device, get_answer_token_id, logit_lens_diff
from circuit_conflict.dataset import load_prompts
from circuit_conflict.metrics import detect_phase_transition, phase_transition_stats

DEVICE = get_device()
RESULTS_DIR = Path('../data/results/phase2')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Device: {DEVICE}')

In [ ]:
model = load_model(DEVICE)
df_all = load_prompts()
df_conflict = df_all[df_all['is_conflict']].reset_index(drop=True)
df_unamb    = df_all[~df_all['is_conflict']].reset_index(drop=True)
print(f'Conflict prompts   : {len(df_conflict)}')
print(f'Unambiguous prompts: {len(df_unamb)}')

## 1. Compute logit-lens curves — conflict prompts

In [ ]:
summary_rows = []

for _, row in tqdm(df_conflict.iterrows(), total=len(df_conflict), desc='Phase 2 conflict'):
    pid = row['prompt_id']
    try:
        tok_A = get_answer_token_id(model, row['answer_A'])
        tok_B = get_answer_token_id(model, row['answer_B'])
    except ValueError as e:
        print(f'  Skipping {pid}: {e}')
        continue

    tokens = model.to_tokens(row['prompt_text'])
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)

    curve = logit_lens_diff(model, cache, tok_A, tok_B)
    transition = detect_phase_transition(curve)

    np.save(RESULTS_DIR / f'layer_curves_{pid}.npy', curve)

    summary_rows.append({
        'prompt_id': pid,
        'category': row['category'],
        'ground_truth': row['ground_truth'],
        'phase_transition_layer': transition,
        'final_logit_diff': float(curve[-1]),
        'curve_min': float(curve.min()),
        'curve_max': float(curve.max()),
        'n_sign_changes': int(np.sum(np.diff(np.sign(curve)) != 0)),
    })

df_phase2 = pd.DataFrame(summary_rows)
df_phase2.to_csv(RESULTS_DIR / 'phase2_summary.csv', index=False)
print(f'Saved summary for {len(df_phase2)} conflict prompts')
df_phase2.head()

## 2. Compute logit-lens curves — unambiguous prompts (for comparison)

In [ ]:
unamb_rows = []

for _, row in tqdm(df_unamb.iterrows(), total=len(df_unamb), desc='Phase 2 unambiguous'):
    pid = row['prompt_id']
    try:
        tok_A = get_answer_token_id(model, row['answer_A'])
        tok_B = get_answer_token_id(model, row['answer_B'])
    except ValueError:
        continue

    tokens = model.to_tokens(row['prompt_text'])
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)

    curve = logit_lens_diff(model, cache, tok_A, tok_B)
    np.save(RESULTS_DIR / f'layer_curves_{pid}.npy', curve)
    unamb_rows.append({'prompt_id': pid, 'category': row['category'], 'curve': curve})

print(f'Computed {len(unamb_rows)} unambiguous curves')

## 3. Phase transition statistics by category

In [ ]:
stats_by_cat = {}
for cat in ['A', 'B', 'C']:
    transitions = df_phase2[df_phase2.category == cat]['phase_transition_layer'].tolist()
    st = phase_transition_stats(transitions)
    stats_by_cat[cat] = st
    print(f'Category {cat}: mean={st["mean"]:.2f} ± {st["std"]:.2f} '
          f'(detected in {st["detection_rate"]:.0%} of prompts)')

# Overall
all_transitions = df_phase2['phase_transition_layer'].tolist()
st_all = phase_transition_stats(all_transitions)
stats_by_cat['All'] = st_all
print(f'Overall: mean={st_all["mean"]:.2f} ± {st_all["std"]:.2f}')

import json
with open(RESULTS_DIR / 'phase_transition_stats.json', 'w') as f:
    json.dump(stats_by_cat, f, indent=2)
print('\nPhase transition stats saved.')

## 4. Logit-lens curves by category (conflict vs. unambiguous)

In [ ]:
N_LAYERS = model.cfg.n_layers
layer_x = np.arange(N_LAYERS)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, cat in zip(axes, ['A', 'B', 'C']):
    # Conflict curves
    conflict_pids = df_phase2[df_phase2.category == cat]['prompt_id'].tolist()
    conflict_curves = []
    for pid in conflict_pids:
        p = RESULTS_DIR / f'layer_curves_{pid}.npy'
        if p.exists():
            conflict_curves.append(np.load(p))

    # Unambiguous curves
    unamb_curves_cat = [
        u['curve'] for u in unamb_rows if u['category'] == cat
    ]

    def plot_mean_band(ax, curves, color, label):
        if not curves:
            return
        arr = np.stack(curves)  # (n, n_layers)
        mean = arr.mean(axis=0)
        std  = arr.std(axis=0)
        ax.plot(layer_x, mean, color=color, lw=2, label=label)
        ax.fill_between(layer_x, mean - std, mean + std, color=color, alpha=0.2)

    plot_mean_band(ax, conflict_curves, 'tab:red', 'Conflict')
    plot_mean_band(ax, unamb_curves_cat, 'tab:blue', 'Unambiguous')
    ax.axhline(0, color='black', lw=0.8, linestyle='--')

    # Mark mean phase transition
    st = stats_by_cat.get(cat, {})
    if st.get('mean') and not np.isnan(st['mean']):
        ax.axvline(st['mean'], color='tab:red', linestyle=':', lw=1.5,
                   label=f'Mean transition L{st["mean"]:.1f}')

    ax.set_title(f'Category {cat}')
    ax.set_xlabel('Layer')
    ax.set_ylabel('Logit diff (A − B)')
    ax.set_xticks(layer_x)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Logit-Lens Curves: Conflict vs. Unambiguous Prompts', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/02_logit_lens_curves.png', dpi=150)
plt.show()

## 5. Phase transition layer distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = {'A': 'tab:blue', 'B': 'tab:orange', 'C': 'tab:green'}

for cat in ['A', 'B', 'C']:
    transitions = [
        t for t in df_phase2[df_phase2.category == cat]['phase_transition_layer']
        if t is not None and not (isinstance(t, float) and np.isnan(t))
    ]
    if transitions:
        ax.hist(transitions, bins=range(0, N_LAYERS + 2),
                alpha=0.6, label=f'Cat {cat} (n={len(transitions)})',
                color=colors[cat], edgecolor='white')

ax.set_xlabel('Phase Transition Layer')
ax.set_ylabel('Count')
ax.set_title('Phase Transition Layer Distribution by Category')
ax.set_xticks(range(N_LAYERS))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/02_phase_transition_distribution.png', dpi=150)
plt.show()

print('\nPhase 2 complete. ✓')